# ROC Curve & AUC (Area Under the Curve)

## Overview
The **ROC (Receiver Operating Characteristic)** curve plots the True Positive Rate (Recall)
against the False Positive Rate at every possible decision threshold.
The **AUC** summarizes the entire curve as a single number.

### Key Definitions:
$$TPR = Recall = \\frac{TP}{TP + FN}$$
$$FPR = \\frac{FP}{FP + TN} = 1 - Specificity$$
$$AUC = \\int_0^1 TPR(FPR) \\, d(FPR)$$

### AUC Interpretation:
| AUC | Meaning |
|-----|---------|
| 1.0 | Perfect classifier |
| 0.9 - 1.0 | Excellent |
| 0.8 - 0.9 | Good |
| 0.7 - 0.8 | Fair |
| 0.6 - 0.7 | Poor |
| 0.5 | Random classifier |
| < 0.5 | Worse than random |

### AUC Probabilistic Interpretation:
AUC = probability that the model ranks a **randomly chosen positive**
instance higher than a **randomly chosen negative** instance.

---
### Topics Covered
1. ROC curve anatomy
2. Effect of threshold on TPR and FPR
3. Comparing multiple models on ROC curve
4. AUC probabilistic interpretation
5. Partial AUC
6. Multiclass ROC (OvR)
7. ROC vs PR curve — when to use which
8. Real-world offline-safe dataset
9. Cross-validated AUC
10. DeLong test — comparing two AUCs


## 1. Import Libraries

In [ ]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.metrics import (
    roc_curve, roc_auc_score, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#E67E22',
    'success':   '#27AE60',
    'danger':    '#E74C3C',
    'neutral':   '#7F8C8D',
    'tertiary':  '#9B59B6',
}
MODEL_COLORS = ['#2E86AB','#27AE60','#E67E22','#9B59B6','#E74C3C','#1ABC9C']

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

print('Libraries imported successfully!')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

## 2. ROC Curve Anatomy

In [ ]:
np.random.seed(42)
X, y = make_classification(
    n_samples=600, n_features=10, n_informative=6,
    n_redundant=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_sc, y_train)
y_prob = model.predict_proba(X_test_sc)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_val = roc_auc_score(y_test, y_prob)

# Find optimal threshold (Youden's J = TPR - FPR)
j_scores  = tpr - fpr
best_idx  = np.argmax(j_scores)
best_thresh = thresholds[best_idx]
best_fpr    = fpr[best_idx]
best_tpr    = tpr[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC curve with annotations
axes[0].plot(fpr, tpr, color=COLORS['primary'], linewidth=2.5,
             label=f'ROC Curve (AUC = {auc_val:.4f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1.5, label='Random (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.12, color=COLORS['primary'])
axes[0].scatter(best_fpr, best_tpr, color=COLORS['danger'], s=200,
                zorder=5, label=f'Optimal thresh={best_thresh:.2f}\n(Youden J)')
axes[0].scatter(0, 1, color=COLORS['success'], s=200, marker='*',
                zorder=5, label='Perfect classifier')

# Annotate corners
axes[0].annotate('All Negative\n(thresh=1)', xy=(0,0), xytext=(0.1,0.05),
                 fontsize=8, color=COLORS['neutral'],
                 arrowprops=dict(arrowstyle='->', color=COLORS['neutral']))
axes[0].annotate('All Positive\n(thresh=0)', xy=(1,1), xytext=(0.7,0.85),
                 fontsize=8, color=COLORS['neutral'],
                 arrowprops=dict(arrowstyle='->', color=COLORS['neutral']))

axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=11)
axes[0].set_title('ROC Curve — Full Anatomy')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-0.02, 1.02)
axes[0].set_ylim(-0.02, 1.05)

# Threshold vs TPR/FPR
axes[1].plot(thresholds, tpr[:-1], color=COLORS['success'],
             linewidth=2, label='TPR (Recall)')
axes[1].plot(thresholds, fpr[:-1], color=COLORS['danger'],
             linewidth=2, label='FPR')
axes[1].plot(thresholds, tpr[:-1]-fpr[:-1], color=COLORS['tertiary'],
             linewidth=2, linestyle='--', label="Youden's J (TPR-FPR)")
axes[1].axvline(best_thresh, color='black', linestyle=':',
                linewidth=1.8, label=f'Optimal thresh={best_thresh:.2f}')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Rate')
axes[1].set_title("Threshold vs TPR / FPR / Youden's J")
axes[1].legend(fontsize=9)

plt.suptitle('ROC Curve Anatomy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'AUC                   : {auc_val:.4f}')
print(f"Youden's J optimal    : threshold={best_thresh:.4f}")
print(f'At optimal threshold  : TPR={best_tpr:.4f}  FPR={best_fpr:.4f}')

## 3. Effect of Threshold on ROC Operating Point

In [ ]:
thresholds_show = [0.1, 0.3, 0.5, 0.7, 0.9]
colors_t = plt.cm.plasma(np.linspace(0.1, 0.9, len(thresholds_show)))

plt.figure(figsize=(9, 7))
plt.plot(fpr, tpr, color=COLORS['primary'], linewidth=2.5,
         label=f'ROC (AUC={auc_val:.3f})', zorder=1)
plt.plot([0,1],[0,1],'k--', linewidth=1.2)
plt.fill_between(fpr, tpr, alpha=0.08, color=COLORS['primary'])

for t, c in zip(thresholds_show, colors_t):
    y_t   = (y_prob >= t).astype(int)
    tn, fp, fn, tp_v = confusion_matrix(y_test, y_t).ravel()
    tpr_t = tp_v / (tp_v + fn) if (tp_v+fn) > 0 else 0
    fpr_t = fp  / (fp  + tn) if (fp +tn) > 0 else 0
    plt.scatter(fpr_t, tpr_t, color=c, s=200, zorder=5,
                edgecolors='black', linewidths=1.2,
                label=f'thresh={t}  TPR={tpr_t:.2f} FPR={fpr_t:.2f}')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Operating Points on ROC Curve at Different Thresholds')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Moving threshold LOWER -> TPR increases, FPR increases (moves up-right)')
print('Moving threshold HIGHER -> TPR decreases, FPR decreases (moves down-left)')
print('The optimal operating point depends on your business requirements.')

## 4. Comparing Multiple Models on ROC Curve

In [ ]:
np.random.seed(42)
X_cmp, y_cmp = make_classification(
    n_samples=700, n_features=15, n_informative=8,
    n_redundant=4, random_state=42
)
X_cmp_tr, X_cmp_te, y_cmp_tr, y_cmp_te = train_test_split(
    X_cmp, y_cmp, test_size=0.2, random_state=42, stratify=y_cmp
)
sc_cmp = StandardScaler()
X_cmp_tr_sc = sc_cmp.fit_transform(X_cmp_tr)
X_cmp_te_sc = sc_cmp.transform(X_cmp_te)

models_cmp = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest':       RandomForestClassifier(100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(100, learning_rate=0.1, random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random (Baseline)':   DummyClassifier(strategy='stratified', random_state=42),
}

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
auc_results = {}

for (name, clf), color in zip(models_cmp.items(), MODEL_COLORS):
    clf.fit(X_cmp_tr_sc, y_cmp_tr)
    prob = clf.predict_proba(X_cmp_te_sc)[:, 1]
    fpr_m, tpr_m, _ = roc_curve(y_cmp_te, prob)
    auc_m = roc_auc_score(y_cmp_te, prob)
    auc_results[name] = auc_m
    ls = '--' if 'Random' in name else '-'
    axes[0].plot(fpr_m, tpr_m, color=color, linewidth=2,
                 linestyle=ls, label=f'{name} (AUC={auc_m:.3f})')

axes[0].plot([0,1],[0,1],'k:', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(fontsize=8)

# AUC bar chart
sorted_auc = dict(sorted(auc_results.items(), key=lambda x: x[1], reverse=True))
bars = axes[1].barh(list(sorted_auc.keys()), list(sorted_auc.values()),
                    color=MODEL_COLORS[:len(sorted_auc)],
                    edgecolor='white', alpha=0.85)
axes[1].axvline(0.5, color='black', linestyle='--',
                linewidth=1.2, label='Random baseline')
axes[1].set_xlim(0.4, 1.05)
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_title('AUC Ranking')
for bar, val in zip(bars, sorted_auc.values()):
    axes[1].text(val + 0.005,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontweight='bold')
axes[1].legend()

plt.suptitle('Multi-Model ROC Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Model ranking by AUC:')
for name, auc_v in sorted(auc_results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:<25} : {auc_v:.4f}')

## 5. AUC Probabilistic Interpretation

In [ ]:
np.random.seed(42)
n_experiments = 5000
pos_probs = y_prob[y_test == 1]
neg_probs = y_prob[y_test == 0]

# Randomly sample one positive and one negative
pos_samples = np.random.choice(pos_probs, n_experiments)
neg_samples = np.random.choice(neg_probs, n_experiments)
empirical_auc = np.mean(pos_samples > neg_samples)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Score distributions
axes[0].hist(neg_probs, bins=25, alpha=0.7, color=COLORS['danger'],
             edgecolor='white', label='Negative class scores', density=True)
axes[0].hist(pos_probs, bins=25, alpha=0.7, color=COLORS['success'],
             edgecolor='white', label='Positive class scores', density=True)
axes[0].axvline(0.5, color='black', linestyle='--',
                linewidth=1.5, label='Default threshold=0.5')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distributions by Class\nMore separation = higher AUC')
axes[0].legend(fontsize=9)

# Empirical AUC verification
diffs = pos_samples - neg_samples
axes[1].hist(diffs, bins=40, color=COLORS['primary'],
             edgecolor='white', alpha=0.85, density=True)
axes[1].axvline(0, color=COLORS['danger'], linewidth=2,
                linestyle='--', label='P(pos) = P(neg)')
axes[1].fill_betweenx(
    [0, axes[1].get_ylim()[1] if axes[1].get_ylim()[1] > 0 else 3],
    0, diffs.max(),
    alpha=0.15, color=COLORS['success'],
    label=f'P(pos score > neg score)\n= Empirical AUC = {empirical_auc:.4f}'
)
axes[1].set_xlabel('pos_score - neg_score')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Probabilistic AUC Interpretation\n'
                  f'sklearn AUC={auc_val:.4f}  Empirical={empirical_auc:.4f}')
axes[1].legend(fontsize=9)

plt.suptitle('AUC = P(positive ranked higher than negative)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'sklearn AUC              : {auc_val:.4f}')
print(f'Empirical AUC ({n_experiments} pairs)  : {empirical_auc:.4f}')
print(f'Difference               : {abs(auc_val-empirical_auc):.4f}')
print('They match closely -- confirming the probabilistic interpretation.')

## 6. Partial AUC — Restricting the FPR Range

In [ ]:
from sklearn.metrics import roc_auc_score

# Partial AUC — useful when only low FPR region matters
# e.g. medical screening where FPR must stay < 20%
max_fpr_values = [0.1, 0.2, 0.3, 0.5, 1.0]

plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color=COLORS['primary'], linewidth=2.5,
         label=f'Full ROC (AUC={auc_val:.4f})')
plt.plot([0,1],[0,1],'k--', linewidth=1.2)

fill_colors = ['#2ECC71','#F39C12','#E74C3C','#9B59B6','#2E86AB']
pauc_results = []
for max_fpr, fc in zip(max_fpr_values, fill_colors):
    pauc = roc_auc_score(y_test, y_prob, max_fpr=max_fpr)
    pauc_results.append((max_fpr, pauc))
    mask = fpr <= max_fpr
    plt.fill_between(fpr[mask], tpr[mask], alpha=0.25, color=fc,
                     label=f'pAUC (FPR<={max_fpr}) = {pauc:.4f}')
    plt.axvline(max_fpr, color=fc, linewidth=1, linestyle=':')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Partial AUC — Restricting the FPR Region\n'
          'Use when low FPR is critical (e.g. medical screening)')
plt.legend(fontsize=8)
plt.xlim(-0.02, 1.02)
plt.tight_layout()
plt.show()

print('Partial AUC results:')
for max_fpr, pauc in pauc_results:
    print(f'  max_fpr={max_fpr:.1f}  pAUC={pauc:.4f}')
print()
print('Use partial AUC when your application requires FPR to stay')
print('below a specific threshold (e.g. regulatory constraint).')

## 7. Multiclass ROC — One-vs-Rest

In [ ]:
np.random.seed(42)
n_cls = 4
X_mc, y_mc = make_classification(
    n_samples=600, n_features=15, n_informative=8,
    n_redundant=3, n_classes=n_cls, n_clusters_per_class=1,
    random_state=42
)
X_mc_tr, X_mc_te, y_mc_tr, y_mc_te = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=42, stratify=y_mc
)
sc_mc = StandardScaler()
X_mc_tr_sc = sc_mc.fit_transform(X_mc_tr)
X_mc_te_sc = sc_mc.transform(X_mc_te)

clf_mc = LogisticRegression(max_iter=500, random_state=42,
                             multi_class='ovr')
clf_mc.fit(X_mc_tr_sc, y_mc_tr)
y_mc_prob = clf_mc.predict_proba(X_mc_te_sc)

# Binarize labels for OvR ROC
y_mc_bin = label_binarize(y_mc_te, classes=list(range(n_cls)))

class_names_mc = [f'Class {k}' for k in range(n_cls)]
colors_mc = [COLORS['primary'], COLORS['success'],
             COLORS['secondary'], COLORS['danger']]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

auc_per_class = []
for k, (name, color) in enumerate(zip(class_names_mc, colors_mc)):
    fpr_k, tpr_k, _ = roc_curve(y_mc_bin[:, k], y_mc_prob[:, k])
    auc_k = roc_auc_score(y_mc_bin[:, k], y_mc_prob[:, k])
    auc_per_class.append(auc_k)
    axes[0].plot(fpr_k, tpr_k, color=color, linewidth=2,
                 label=f'{name} (AUC={auc_k:.3f})')

axes[0].plot([0,1],[0,1],'k--', linewidth=1.2)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Multiclass ROC — One-vs-Rest (OvR)')
axes[0].legend(fontsize=9)

# Macro and weighted AUC
auc_macro    = roc_auc_score(y_mc_te, y_mc_prob,
                              multi_class='ovr', average='macro')
auc_weighted = roc_auc_score(y_mc_te, y_mc_prob,
                              multi_class='ovr', average='weighted')

bars_mc = axes[1].bar(class_names_mc + ['Macro\nAUC', 'Weighted\nAUC'],
                      auc_per_class + [auc_macro, auc_weighted],
                      color=colors_mc + [COLORS['neutral'], COLORS['tertiary']],
                      edgecolor='white', alpha=0.85)
axes[1].axhline(0.5, color='black', linestyle='--',
                linewidth=1.2, label='Random baseline')
axes[1].set_ylim(0.4, 1.1)
axes[1].set_ylabel('AUC')
axes[1].set_title('Per-Class AUC + Macro + Weighted')
for bar, val in zip(bars_mc, auc_per_class + [auc_macro, auc_weighted]):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold')
axes[1].legend()

plt.suptitle('Multiclass ROC-AUC (One-vs-Rest)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Per-class AUC  : {[round(a,4) for a in auc_per_class]}')
print(f'Macro AUC      : {auc_macro:.4f}')
print(f'Weighted AUC   : {auc_weighted:.4f}')

## 8. ROC vs PR Curve — When to Use Which

In [ ]:
np.random.seed(42)

# Balanced dataset
X_bal, y_bal = make_classification(
    n_samples=600, n_features=10, n_informative=6,
    weights=[0.5, 0.5], random_state=42
)
# Imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=600, n_features=10, n_informative=6,
    weights=[0.95, 0.05], random_state=42
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for row, (X_ds, y_ds, ds_name) in enumerate([
    (X_bal, y_bal, 'Balanced (50/50)'),
    (X_imb, y_imb, 'Imbalanced (95/5)'),
]):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_ds, y_ds, test_size=0.2, random_state=42, stratify=y_ds
    )
    sc_ds = StandardScaler()
    X_tr_sc = sc_ds.fit_transform(X_tr)
    X_te_sc = sc_ds.transform(X_te)

    clf_ds = LogisticRegression(max_iter=500, random_state=42,
                                 class_weight='balanced')
    clf_ds.fit(X_tr_sc, y_tr)
    prob_ds = clf_ds.predict_proba(X_te_sc)[:, 1]

    fpr_ds, tpr_ds, _  = roc_curve(y_te, prob_ds)
    prec_ds, rec_ds, _ = precision_recall_curve(y_te, prob_ds)
    auc_ds  = roc_auc_score(y_te, prob_ds)
    ap_ds   = average_precision_score(y_te, prob_ds)
    baseline_ds = y_te.mean()

    axes[row][0].plot(fpr_ds, tpr_ds, color=COLORS['primary'], linewidth=2.5)
    axes[row][0].plot([0,1],[0,1],'k--', linewidth=1.2)
    axes[row][0].fill_between(fpr_ds, tpr_ds, alpha=0.12, color=COLORS['primary'])
    axes[row][0].set_xlabel('FPR'); axes[row][0].set_ylabel('TPR')
    axes[row][0].set_title(f'ROC Curve — {ds_name}\nAUC={auc_ds:.4f}')

    axes[row][1].plot(rec_ds, prec_ds, color=COLORS['secondary'], linewidth=2.5)
    axes[row][1].axhline(baseline_ds, color='black', linestyle='--',
                         linewidth=1.2,
                         label=f'Random baseline={baseline_ds:.2f}')
    axes[row][1].fill_between(rec_ds, prec_ds, alpha=0.12, color=COLORS['secondary'])
    axes[row][1].set_xlabel('Recall'); axes[row][1].set_ylabel('Precision')
    axes[row][1].set_title(f'PR Curve — {ds_name}\nAP={ap_ds:.4f}')
    axes[row][1].legend(fontsize=9)

plt.suptitle('ROC vs PR Curve — Balanced vs Imbalanced Classes',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('ROC curve: optimistic on imbalanced data (large TN pool makes FPR look small)')
print('PR curve : more informative on imbalanced data (focuses on positive class)')
print()
print('Rule: if positive class < 10% of data -> use PR curve / Average Precision')

## 9. Real-World Offline-Safe Dataset — Disease Screening

In [ ]:
np.random.seed(42)
n_pts = 800

X_dis, y_dis = make_classification(
    n_samples=n_pts, n_features=12, n_informative=7,
    n_redundant=3, weights=[0.80, 0.20],
    random_state=42
)
feat_names_dis = ['Age','BMI','BloodPres','Glucose','Cholesterol',
                  'Insulin','HeartRate','CRP','Albumin',
                  'CreatineKinase','Troponin','Hemoglobin']

print(f'Dataset shape  : {X_dis.shape}')
print(f'Class balance  : {np.bincount(y_dis)}  (0=Healthy, 1=At Risk)')

X_dis_tr, X_dis_te, y_dis_tr, y_dis_te = train_test_split(
    X_dis, y_dis, test_size=0.2, random_state=42, stratify=y_dis
)

dis_models = {
    'Logistic Reg':      LogisticRegression(max_iter=500, class_weight='balanced',
                                             random_state=42),
    'Random Forest':     RandomForestClassifier(100, class_weight='balanced',
                                                random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(100, learning_rate=0.1,
                                                     random_state=42),
}

sc_dis    = StandardScaler()
X_dis_tr_sc = sc_dis.fit_transform(X_dis_tr)
X_dis_te_sc = sc_dis.transform(X_dis_te)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
auc_dis_results = {}

for (name, clf), color in zip(dis_models.items(),
                               [COLORS['primary'], COLORS['success'],
                                COLORS['secondary']]):
    clf.fit(X_dis_tr_sc, y_dis_tr)
    prob_dis = clf.predict_proba(X_dis_te_sc)[:, 1]
    fpr_d, tpr_d, _  = roc_curve(y_dis_te, prob_dis)
    prec_d, rec_d, _ = precision_recall_curve(y_dis_te, prob_dis)
    auc_d  = roc_auc_score(y_dis_te, prob_dis)
    ap_d   = average_precision_score(y_dis_te, prob_dis)
    auc_dis_results[name] = (auc_d, ap_d)

    axes[0].plot(fpr_d, tpr_d, color=color, linewidth=2,
                 label=f'{name} (AUC={auc_d:.3f})')
    axes[1].plot(rec_d, prec_d, color=color, linewidth=2,
                 label=f'{name} (AP={ap_d:.3f})')

axes[0].plot([0,1],[0,1],'k--', linewidth=1.2)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves — Disease Screening')
axes[0].legend(fontsize=9)

baseline_dis = y_dis_te.mean()
axes[1].axhline(baseline_dis, color='black', linestyle='--',
                linewidth=1.2, label=f'Baseline={baseline_dis:.2f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves — Disease Screening')
axes[1].legend(fontsize=9)

plt.suptitle('Disease Screening — ROC & PR Curves (3 Models)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Model Results:')
print(f'  {"Model":<22} {"ROC-AUC":>10} {"Avg Prec":>10}')
print('  ' + '-'*45)
for name, (auc_v, ap_v) in auc_dis_results.items():
    print(f'  {name:<22} {auc_v:>10.4f} {ap_v:>10.4f}')

## 10. Cross-Validated AUC

In [ ]:
cv_models = {
    'Logistic Regression': Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('sc',  StandardScaler()),
        ('clf', RandomForestClassifier(100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('sc',  StandardScaler()),
        ('clf', GradientBoostingClassifier(100, learning_rate=0.1, random_state=42))
    ]),
}

cv_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, pipe in cv_models.items():
    scores = cross_val_score(
        pipe, X_cmp, y_cmp,
        cv=cv_kf, scoring='roc_auc'
    )
    cv_results[name] = scores
    print(f'{name:<25}: mean={scores.mean():.4f}  std={scores.std():.4f}'
          f'  [{" ".join([f"{s:.3f}" for s in scores])}]')

# Boxplot
plt.figure(figsize=(9, 5))
bp = plt.boxplot(list(cv_results.values()),
                 labels=list(cv_results.keys()),
                 patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'],
                         [COLORS['primary'], COLORS['success'],
                          COLORS['secondary']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
plt.ylabel('ROC-AUC')
plt.title('5-Fold Cross-Validated AUC Distribution')
plt.tight_layout()
plt.show()

print()
print('Cross-validated AUC is more reliable than single train/test split.')
print('Use mean +/- std to compare models fairly.')

## 11. Final Summary & Key Takeaways

In [ ]:
print('=' * 66)
print('            ROC CURVE & AUC - KEY TAKEAWAYS')
print('=' * 66)
takeaways = [
    ('ROC curve',        'TPR vs FPR at every threshold'),
    ('AUC range',        '0.5 = random, 1.0 = perfect, < 0.5 = worse than random'),
    ('Probabilistic',    'AUC = P(pos scored higher than neg)'),
    ('Threshold-free',   'AUC evaluates ranking ability, not specific threshold'),
    ("Youden's J",       'TPR-FPR maximized at optimal threshold'),
    ('Partial AUC',      'Restrict FPR range for domain-specific constraints'),
    ('Multiclass',       'OvR binarization + macro/weighted average'),
    ('ROC vs PR',        'Use PR curve when positive class is rare (< 10%)'),
    ('Imbalanced',       'ROC-AUC can be misleadingly high; PR-AUC more honest'),
    ('Cross-validate',   'Always report CV AUC mean +/- std, not single split'),
    ('class_weight',     'Use balanced when classes are imbalanced'),
    ('Model comparison', 'Higher AUC = better ranking; compare with error bars'),
]
for topic, detail in takeaways:
    print(f'  OK  {topic:<18}  ->  {detail}')
print('=' * 66)

---
## Practice Exercises

1. **Custom threshold** — using Youden's J, pick the optimal threshold and compare the confusion matrix to the default 0.5.
2. **Partial AUC** — set `max_fpr=0.1` and rank models differently — does ranking change?
3. **Imbalanced dataset** — generate a 99/1 split and compare ROC-AUC vs PR-AUC across models.
4. **DeLong test** — install `scipy` and implement a permutation test to check if two AUCs differ significantly.
5. **Bootstrap AUC CI** — resample test set 1000 times and compute 95% confidence interval for AUC.
6. **Multiclass OvO** — use `multi_class='ovo'` instead of `'ovr'` and compare per-class AUCs.

---
*Notebook created for the ML Repository — ROC Curve & AUC module.*